# Implémentation et Vérification du Routage Fallback (Réaffectation)

<a href="https://colab.research.google.com/github/JunSuzukiJapan/SynapticRouter/blob/main/notebooks/17_routing_fallback_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Aperçu
Dans ce notebook, nous implémentons **Routing Fallback** pour résoudre fondamentalement le problème de Lazy Routing (la paresse du routeur) dans SRA (Synaptic Routing Architecture).

Dans l'expérience précédente (Notebook 16), une simple limite de capacité (tronquage des jetons de débordement) provoquait une perte d'informations (Drop) et avait des limites pour éliminer complètement les synapses mortes ou améliorer la précision.
Dans cette expérience, nous implémentons une **logique qui transfère (replie) de force les jetons dépassant la capacité du routeur vers une « synapse avec de l'espace libre dont la probabilité est la deuxième ou la plus élevée »**, et vérifions les effets suivants :

1. Le **taux de synapses mortes devient-il complètement 0 %** ?
2. La **précision globale (perte CE) du modèle s'améliore-t-elle** grâce à une perte d'information réduite ?

In [ ]:
# Setup for running on Google Colab
import os
import sys

if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    sys.path.append(os.getcwd())
    sys.path.append(os.path.join(os.getcwd(), 'src'))
    print("Setup completed on Google Colab.")
else:
    sys.path.append(os.path.abspath('..'))
    sys.path.append(os.path.join(os.path.abspath('..'), 'src'))
    print("Running locally.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import matplotlib.pyplot as plt

from src.sra_gpu_models import MoESRAModel, MoESRABlock
from src.sra_experiment import make_optimizer, load_balance_loss
from src.constants import PAD, BOS, EOS

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# ===== Task definitions (excerpt) =====
VOCAB_SIZE = 128
def encode(text): return [BOS] + [ord(c) for c in text] + [EOS]
def pad_to(seq, length): return seq[:length] + [PAD] * max(0, length - len(seq))

WORDS = ["hello", "world", "python", "learn", "great", "smart", "open", "code"]
VARS = ["x", "y", "z", "n", "val", "res"]
BASES = ['A', 'T', 'G', 'C']
COMP  = {'A':'T', 'T':'A', 'G':'C', 'C':'G'}

def nl_upper(): w = random.choice(WORDS); return w, w.upper()
def code_indent(): v = random.choice(VARS); n = random.randint(1,9); return f"return {v} + {n}", f"    return {v} + {n}"
def math_add(): a, b = random.randint(1,19), random.randint(1,19); return f"{a}+{b}=", str(a+b)
def dna_complement(): s = ''.join(random.choices(BASES, k=5)); return s, ''.join(COMP[c] for c in s)

ALL_TASKS = {
    "nl_upper": nl_upper, "code_indent": code_indent,
    "math_add": math_add, "dna_complement": dna_complement
}

MAX_SEQ_LEN = 24
DIM = 64
LAYERS = 2
K = 2
SYN_HIDDEN = 128

def make_multitask_batch(tasks, batch_size):
    xs, ys = [], []
    for _ in range(batch_size):
        task_name = random.choice(list(tasks.keys()))
        inp_str, out_str = tasks[task_name]()
        xs.append(pad_to(encode(inp_str), MAX_SEQ_LEN))
        ys.append(pad_to(encode(out_str), MAX_SEQ_LEN))
    return torch.tensor(xs, dtype=torch.long, device=device), torch.tensor(ys, dtype=torch.long, device=device)

In [ ]:
def calculate_routing_metrics(model, num_synapses, use_fallback=False, capacity_factor=None, samples=100):
    model.eval()
    all_usage_counts = torch.zeros(num_synapses, device=device)
    total_tokens = 0
    all_probs = []
    
    with torch.no_grad():
        for _ in range(samples):
            x, y = make_multitask_batch(ALL_TASKS, 128)
            y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
            valid_mask = (torch.cat([x, y_in], dim=1) != PAD)
            
            kwargs = {}
            if capacity_factor is not None:
                kwargs['capacity_factor'] = capacity_factor
            
            _, router_logits, _ = model(x, y_in, **kwargs)
            logits = router_logits[-1] # (B, T, num_synapses)
            probs = F.softmax(logits, dim=-1)
            
            # Entropy
            valid_probs = probs[valid_mask]
            all_probs.append(valid_probs)
            
            # Use top-K to measure usage
            _, topk = torch.topk(logits, K, dim=-1)
            valid_topk = topk[valid_mask]
            for k_idx in range(K):
                all_usage_counts.index_add_(0, valid_topk[:, k_idx], torch.ones(valid_topk.size(0), device=device))
            total_tokens += valid_topk.size(0)
            
    # Routing Entropy
    cat_probs = torch.cat(all_probs, dim=0) # (Total_Tokens, num_synapses)
    mean_probs = cat_probs.mean(dim=0)
    routing_entropy = -(mean_probs * torch.log(mean_probs + 1e-9)).sum().item()
    
    # Dead Synapse Ratio (< 1% of expected uniform usage)
    expected_usage = (total_tokens * K) / num_synapses
    dead_threshold = expected_usage * 0.01
    dead_synapses = (all_usage_counts < dead_threshold).sum().item()
    dead_ratio = (dead_synapses / num_synapses) * 100
    
    return routing_entropy, dead_ratio

In [ ]:
class MoESRABlockWithFallback(MoESRABlock):
    def forward(self, h, dense=False, key_padding_mask=None, encoder_len=0, capacity_factor=None, use_fallback=False):
        base = h
        B, T, D = h.shape
        
        # 1. Shared Attention
        attn_mask = torch.zeros((T, T), dtype=torch.bool, device=h.device)
        if encoder_len > 0:
            attn_mask[:encoder_len, encoder_len:] = True
            future_target = torch.triu(torch.ones((T - encoder_len, T - encoder_len), dtype=torch.bool, device=h.device), diagonal=1)
            attn_mask[encoder_len:, encoder_len:] = future_target
            
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False)
        h = self.norm1(h + attn_out)
        
        # 2. MoE Routing
        h_routed = h
        h_routed = self.norm2(h_routed)
        
        max_fallback = 4 if use_fallback else 0
        N = min(self.num_synapses, self.k + max_fallback)
        k_override = self.num_synapses if dense else N
        
        idx_all, weights_all, logits = self.router(h_routed, k_override=k_override)
        
        h_flat = h_routed.view(B*T, D)
        out_flat = torch.zeros_like(h_flat)
        
        if capacity_factor is not None:
            capacity_limit = int((B * T * capacity_factor) / self.num_synapses)
        else:
            capacity_limit = B * T
            
        # Fallback & Capacity Logic
        idx_flat = idx_all.view(B*T, -1)
        weights_flat = weights_all.view(B*T, -1)
        
        expert_counts = torch.zeros(self.num_synapses, dtype=torch.long, device=h.device)
        final_expert_mask = torch.zeros(self.num_synapses, B*T, dtype=torch.bool, device=h.device)
        final_expert_weights = torch.zeros(self.num_synapses, B*T, device=h.device)
        
        assigned_k = torch.zeros(B*T, dtype=torch.long, device=h.device)
        
        # Iteratively assign tokens to their top preferences
        for pref in range(N):
            desired_expert = idx_flat[:, pref]
            desired_weight = weights_flat[:, pref]
            
            needs_assignment = (assigned_k < self.k)
            if not needs_assignment.any():
                break
                
            # Vectorized assignment to avoid GPU-CPU syncs (.item())
            # wants_e shape: (num_synapses, B*T)
            experts_range = torch.arange(self.num_synapses, device=h.device).unsqueeze(1)
            wants_e = needs_assignment.unsqueeze(0) & (desired_expert.unsqueeze(0) == experts_range)
            
            # cumulative sum to find which tokens are within capacity
            # cumsum shape: (num_synapses, B*T)
            cumsum = wants_e.long().cumsum(dim=1)
            
            # available capacity for each expert: (num_synapses, 1)
            avail = capacity_limit - expert_counts.unsqueeze(1)
            
            # Accepted mask: (num_synapses, B*T)
            accepted = wants_e & (cumsum <= avail)
            
            final_expert_mask |= accepted
            final_expert_weights = torch.where(accepted, desired_weight.unsqueeze(0), final_expert_weights)
            
            expert_counts += accepted.long().sum(dim=1)
            assigned_k += accepted.long().sum(dim=0)
        
        # Re-normalize weights for tokens that got assigned
        sum_weights = final_expert_weights.sum(dim=0, keepdim=True) + 1e-9
        final_expert_weights = final_expert_weights / sum_weights
            
        for e in range(self.num_synapses):
            mask = final_expert_mask[e]
            if not mask.any(): continue
            
            token_indices = mask.nonzero().squeeze(-1)
            h_sub = h_flat[token_indices]
            
            w1_ex = self.w1[e]
            b1_ex = self.b1[e]
            w2_ex = self.w2[e]
            b2_ex = self.b2[e]
            state_ex = self.state[e]
            
            hidden = F.gelu(torch.matmul(h_sub, w1_ex) + b1_ex)
            expert_out = torch.matmul(hidden, w2_ex) + b2_ex + state_ex
            
            expert_weights = final_expert_weights[e, token_indices]
            out_flat[token_indices] += expert_out * expert_weights.unsqueeze(-1)
            
        out = out_flat.view(B, T, D)
        dummy_syn_outs = [torch.zeros(B, T, D, device=h.device) for _ in range(self.num_synapses)]
        return base + out, logits, dummy_syn_outs

class MoESRAModelWithFallback(MoESRAModel):
    def __init__(self, vocab_size, dim, layers, num_synapses, k, syn_hidden):
        super().__init__(vocab_size, dim, layers, num_synapses, k, syn_hidden)
        self.blocks = nn.ModuleList([MoESRABlockWithFallback(dim, num_synapses, k, syn_hidden) for _ in range(layers)])
        
    def forward(self, x, y_in, dense=False, capacity_factor=None, use_fallback=False):
        seq = torch.cat([x, y_in], dim=1)
        mask = seq == PAD
        segment_ids = torch.cat([torch.zeros_like(x), torch.ones_like(y_in)], dim=1)
        target_rel_pos = torch.cat([torch.zeros_like(x), torch.arange(1, y_in.size(1) + 1, device=seq.device).unsqueeze(0).repeat(x.size(0), 1)], dim=1)
        h = self.embed(seq) + self.pos[:, :seq.size(1)] + self.seg(segment_ids) + self.rel_pos(target_rel_pos)
        
        router_logits = []
        all_synapse_outputs = []
        for block in self.blocks:
            h, logits, syn_outs = block(h, dense=dense, key_padding_mask=mask, encoder_len=x.size(1), capacity_factor=capacity_factor, use_fallback=use_fallback)
            router_logits.append(logits)
            all_synapse_outputs.append(syn_outs)
        return self.out(h[:, x.size(1):]), router_logits, all_synapse_outputs

In [ ]:
def train_model_experiment(num_synapses, epochs=1000, capacity_factor=None, use_fallback=False):
    model = MoESRAModelWithFallback(vocab_size=128, dim=DIM, layers=LAYERS, num_synapses=num_synapses, k=K, syn_hidden=SYN_HIDDEN).to(device)
    optimizer = make_optimizer(model, lr=1e-3)
    model.train()
    
    final_ce_loss = 0.0
    for epoch in range(epochs):
        x, y = make_multitask_batch(ALL_TASKS, 128)
        optimizer.zero_grad()
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        
        outputs, routing_weights, _ = model(x, y_in, capacity_factor=capacity_factor, use_fallback=use_fallback)
        
        ce_loss = F.cross_entropy(outputs.reshape(-1, 128), y.reshape(-1), ignore_index=PAD)
        lb_loss = load_balance_loss(routing_weights) * 0.1
        total_loss = ce_loss + lb_loss
            
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        final_ce_loss = ce_loss.item()
        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{epochs} | CE Loss: {final_ce_loss:.4f}")
            
    return model, final_ce_loss

In [ ]:
num_synapses = 128
epochs = 1000

conditions = [
    {"name": "1. Baseline", "capacity_factor": None, "use_fallback": False},
    {"name": "2. Capacity Limit Only", "capacity_factor": 1.5, "use_fallback": False},
    {"name": "3. Routing Fallback", "capacity_factor": 1.5, "use_fallback": True}
]

results = {}

for cond in conditions:
    print(f"\n--- Training Condition: {cond['name']} ---")
    model, final_loss = train_model_experiment(
        num_synapses=num_synapses, 
        epochs=epochs, 
        capacity_factor=cond["capacity_factor"], 
        use_fallback=cond["use_fallback"]
    )
    
    print(f"Calculating metrics for {cond['name']}...")
    entropy, dead_ratio = calculate_routing_metrics(model, num_synapses, use_fallback=cond["use_fallback"], capacity_factor=cond["capacity_factor"])
    print(f"CE Loss: {final_loss:.4f} | Routing Entropy: {entropy:.3f} | Dead Synapse Ratio: {dead_ratio:.1f}%")
    
    results[cond["name"]] = {
        "loss": final_loss,
        "entropy": entropy,
        "dead_ratio": dead_ratio
    }

In [ ]:
# Visualization of the results
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
names = list(results.keys())
losses = [results[n]["loss"] for n in names]
entropies = [results[n]["entropy"] for n in names]
dead_ratios = [results[n]["dead_ratio"] for n in names]

# Max possible entropy
max_entropy = float(np.log(num_synapses))
colors = ['gray', 'orange', 'green']

bars1 = ax1.bar(names, losses, color=colors)
ax1.set_title("Cross Entropy Loss (Lower is better)")
ax1.set_ylabel("CE Loss")
ax1.tick_params(axis='x', rotation=15)

bars2 = ax2.bar(names, entropies, color=colors)
ax2.axhline(max_entropy, color='red', linestyle='--', label=f'Max Entropy ({max_entropy:.2f})')
ax2.set_title("Routing Entropy (Higher is better/more uniform)")
ax2.set_ylabel("Entropy")
ax2.tick_params(axis='x', rotation=15)
ax2.legend()

bars3 = ax3.bar(names, dead_ratios, color=colors)
ax3.set_title("Dead Synapse Ratio (Lower is better)")
ax3.set_ylabel("Ratio (%)")
ax3.set_ylim(0, 105)
ax3.tick_params(axis='x', rotation=15)

# Display values above the bars
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval, f'{yval:.3f}', ha='center', va='bottom')
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.1, f'{yval:.2f}', ha='center', va='bottom')
for bar in bars3:
    yval = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval:.1f}%', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Discussion
Les résultats expérimentaux nous donnent les informations importantes suivantes sur l’effet du Routing Fallback et les propriétés du MoE.

### 1. Amélioration spectaculaire du ratio de synapses mortes
- **Référence (24,2 %)** -> **Limite de capacité (14,1 %)** -> **Routage de repli (7,8 %)**
- Plutôt que de simplement tronquer les jetons qui débordent, Fallback "les achemine de force vers une autre synapse libre", ce qui réduit plus efficacement les synapses mortes. Il n'atteint pas exactement 0 % car certaines synapses se retrouvent avec une probabilité extrêmement faible à l'initialisation et ne parviennent pas à figurer parmi les choix "top-N (par exemple, 4ème préférence)" d'un jeton (pour atteindre un 0 % parfait, il faudrait combiner cela avec une perte d'entropie).

### 2. Une découverte surprenante sur la perte de CE (précision)
- **Baseline (0,003)** présente la perte la plus faible, tandis que **Capacity Limit (0,037)** et **Routing Fallback (0,039)** présentent des pertes légèrement plus élevées.
- Contrairement à notre prédiction précédente, **"un modèle sans contrainte comme la ligne de base prend le raccourci de tout transférer sur quelques synapses exceptionnelles (en étant paresseux) pour réduire rapidement la perte."**
- Cependant, lorsque les connaissances sont excessivement mélangées dans un petit ensemble de synapses, l'objectif initial du SRA, à savoir un « désapprentissage sécurisé », devient difficile. Autrement dit, la légère augmentation de la Perte due au Fallback (0,039 correspond toujours à une très bonne précision proche de 100%) peut être interprétée positivement comme **"le coût nécessaire payé en échange de l'interdiction de la paresse rapide et de l'obtention d'une grande modularité (Mort 7,8%)."**